# Template notebook

You’re working with **Northstar Desk**, a UK-based subscription software company. Over the last year, the team has handled a steady stream of operational cases: billing and renewals, access and admin requests, reporting issues, integrations, bugs, and occasional performance incidents. The dataset you’ve been given is a snapshot export from Northstar’s case management system: each row is one case, with structured triage fields, outcomes, timings, and a short free-text summary.

Northstar doesn’t want a “fully automated AI that replaces support staff”. They want something more practical: a tool that helps humans **make better decisions, faster**, using the data they already have.

## Your mission

Build a **decision-support prototype** that improves how a frontline or operational user handles cases. Your tool can focus on any part of the workflow, as long as it produces something a real person could use.

Your prototype should support at least one of these outcomes:

- **Understand** what’s coming in (themes, clusters, common issues, emerging patterns)  
- **Prioritise** what to look at first (risk signals, urgency proxies, workload triage)  
- **Route** work more effectively (suggest team/category/subcategory, detect likely escalations)  
- **Resolve** issues faster (surface likely next steps, similar cases, common fixes)  
- **Learn** from outcomes (what tends to lead to long resolution times, escalations, poor CSAT)

You can interpret “decision-support” broadly. The key is that it helps a user do something concrete with the case data.

## Prototype requirement (this is the steer)

Your prototype must be an **interactive tool**, not just analysis:

- A **Gradio app** (recommended), or other small deployable interface.  
- It can be simple: one or two screens, a small number of inputs, and clear outputs.  
- It should demonstrate a realistic workflow: a user provides inputs (e.g. a case summary, category, priority, or a date range) and your tool returns helpful outputs (e.g. routing suggestions, similar cases, risk flags, or a summary of patterns).

A notebook is fine for development, but the end result should include an interface that could plausibly be shared with a non-technical colleague.

## Choose a track (optional)

If you’re having trouble choosing what to build, you can try one of the following tracks:

- **Track 1: Triage assistant** — classify, route, prioritise, or flag risk for new cases.  
- **Track 2: Ops insight tool** — explore trends and spikes, with filters a team lead would use.  
- **Track 3: Similarity + retrieval tool** — find related cases and surface “what worked before”.  
- **Track 4: Process quality tool** — analyse what drives delays/escalations and where process breaks down (optional fairness checks as audit, not decisioning).

Any track is valid. You’re judged on usefulness and clarity, not on picking the “right” one.

## What you’ll present (end of day 2)

1. **Working prototype**  
    Gradio app (or similar) demonstrating the decision-support workflow.  
2. **Short technical summary**  
    Data prep, modelling choices, and why they make sense. Risks, limitations, and interpretability choices.  
3. **Short user-facing story**  
    Who uses the tool, what problem it solves for them, and how they’d use it day-to-day.  
4. **Roadmap**  
    What you’d do next with another week, or if you were building this internally (data, modelling, UX, governance, monitoring).


## Libraries
As always, we'll start by importing the necessary libraries.

In [ ]:
# It's good practice to add comments to explain your code 
import numpy as np
import pandas as pd

In [ ]:
import importlib
import model_utils

importlib.reload(model_utils)

from model_utils import (
    ESCALATION_ROUTING_FEATURES,
    ETHICS_ANALYSIS_COLUMNS,
    FUTURE_TARGETS,
    FUTURE_TEXT_FEATURES,
    PRIMARY_EVALUATION_SLICE,
    ROUTING_FEATURES,
    build_assigned_team_frames_by_slice,
    build_escalation_team_frames,
    evaluate_manual_review_thresholds,
    final_escalation_train_demo_split,
    final_train_demo_split,
    load_case_data,
    make_pipeline,
    train_temporal_classifier,
)

MODEL_TYPES = ["logistic", "decision_tree", "random_forest"]
FINAL_MODEL_TYPE = "logistic"
FINAL_MODEL_SLICE = "solved_only"
DEMO_CASE_SLICE = "all_cases"
MANUAL_REVIEW_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90]
SELECTED_REVIEW_THRESHOLD = 0.70

cases, repair_log = load_case_data()

print(f"Loaded {cases.shape[0]:,} cases from {cases['source_file'].nunique()} files")
print(f"Applied {len(repair_log):,} row repairs")
display(repair_log.value_counts(["source_file", "issue"]).rename("repairs").reset_index())
#display(cases.head())

In [ ]:
assigned_team_frames_by_slice = build_assigned_team_frames_by_slice(cases)
escalation_team_frames = build_escalation_team_frames(cases)

ethics_analysis_frame = cases[
    ["case_id", "status", "escalated", "assigned_team", "escalation_team"]
    + ETHICS_ANALYSIS_COLUMNS
].copy()

split_summary_rows = []
for slice_name, frames in assigned_team_frames_by_slice.items():
    for split_name, split in frames.items():
        split_summary_rows.append(
            {
                "target": "assigned_team",
                "slice": slice_name,
                "split": split_name,
                "rows": len(split["y"]),
                "classes": split["y"].nunique(),
                "files": ", ".join(sorted(split["cases"]["source_file"].unique())),
            }
        )

for split_name, split in escalation_team_frames.items():
    split_summary_rows.append(
        {
            "target": "escalation_team",
            "slice": "escalated_with_known_team",
            "split": split_name,
            "rows": len(split["y"]),
            "classes": split["y"].nunique(),
            "files": ", ".join(sorted(split["cases"]["source_file"].unique())),
        }
    )

print("Temporal split summary")
display(pd.DataFrame(split_summary_rows))
print("Ethics-only columns kept out of model features:", ETHICS_ANALYSIS_COLUMNS)
print("Future modelling candidates:", FUTURE_TARGETS + FUTURE_TEXT_FEATURES)

In [ ]:
from pathlib import Path
import joblib

models = {}
evaluation_rows = []

for slice_name, frames in assigned_team_frames_by_slice.items():
    print("=" * 80)
    print(f"Assigned-team evaluation scenario: {slice_name}")

    for model_type in MODEL_TYPES:
        model, metrics = train_temporal_classifier(
            frames,
            "assigned_team",
            model_type=model_type,
            slice_name=slice_name,
        )
        models[("assigned_team", slice_name, model_type)] = model
        evaluation_rows.append(metrics)

assigned_team_model = models[("assigned_team", PRIMARY_EVALUATION_SLICE, FINAL_MODEL_TYPE)]

# Escalation-team labels only exist for escalated cases, so keep this as a separate routing task.
escalation_candidates = {}
for model_type in MODEL_TYPES:
    escalation_model, escalation_metrics = train_temporal_classifier(
        escalation_team_frames,
        "escalation_team",
        model_type=model_type,
        slice_name="escalated_with_known_team",
    )
    escalation_candidates[model_type] = {
        "model": escalation_model,
        "metrics": escalation_metrics,
    }
    models[("escalation_team", "escalated_with_known_team", model_type)] = escalation_model
    evaluation_rows.append(escalation_metrics)

best_escalation_model_type = max(
    MODEL_TYPES,
    key=lambda model_type: escalation_candidates[model_type]["metrics"]["test_accuracy"],
)
best_escalation_model = escalation_candidates[best_escalation_model_type]["model"]

evaluation_summary = pd.DataFrame(evaluation_rows)
print("\nModel evaluation summary")
display(evaluation_summary)

threshold_summary_rows = []
confidence_results_by_scenario = {}
for scenario_name, frames in assigned_team_frames_by_slice.items():
    scenario_model = models[("assigned_team", scenario_name, FINAL_MODEL_TYPE)]
    scenario_threshold_summary, scenario_confidence_results = evaluate_manual_review_thresholds(
        scenario_model,
        frames,
        thresholds=MANUAL_REVIEW_THRESHOLDS,
    )
    scenario_threshold_summary.insert(0, "scenario", scenario_name)
    threshold_summary_rows.append(scenario_threshold_summary)
    confidence_results_by_scenario[scenario_name] = scenario_confidence_results

threshold_summary = pd.concat(threshold_summary_rows, ignore_index=True)
print("\nManual review threshold summary")
display(threshold_summary)

selected_threshold_summary = threshold_summary[
    threshold_summary["threshold"].eq(SELECTED_REVIEW_THRESHOLD)
].copy()
print(f"\nSelected manual review threshold: {SELECTED_REVIEW_THRESHOLD:.0%}")
display(selected_threshold_summary)

final_train_cases, demo_cases = final_train_demo_split(
    cases,
    slice_name=FINAL_MODEL_SLICE,
    demo_slice_name=DEMO_CASE_SLICE,
)
final_assigned_team_pipeline = make_pipeline(FINAL_MODEL_TYPE)
final_assigned_team_pipeline.fit(
    final_train_cases[ROUTING_FEATURES],
    final_train_cases["assigned_team"],
)

final_escalation_train_cases, final_escalation_demo_cases = final_escalation_train_demo_split(cases)
final_escalation_team_pipeline = make_pipeline(
    best_escalation_model_type,
    feature_columns=ESCALATION_ROUTING_FEATURES,
)
final_escalation_team_pipeline.fit(
    final_escalation_train_cases[ESCALATION_ROUTING_FEATURES],
    final_escalation_train_cases["escalation_team"],
)

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
joblib.dump(final_assigned_team_pipeline, model_dir / "assigned_team_pipeline.joblib")
joblib.dump(final_escalation_team_pipeline, model_dir / "escalation_team_pipeline.joblib")

print(
    f"Saved {FINAL_MODEL_TYPE} assigned-team pipeline trained on Jan-Sept "
    f"({FINAL_MODEL_SLICE}, {len(final_train_cases):,} rows)."
)
print(
    f"Saved {best_escalation_model_type} escalation-team pipeline trained on Jan-Sept "
    f"escalated cases ({len(final_escalation_train_cases):,} rows)."
)
print(
    f"Oct-Dec demo cases are available from the raw data via final_train_demo_split "
    f"({DEMO_CASE_SLICE}, {len(demo_cases):,} rows)."
)
print(
    f"Oct-Dec escalated demo cases with known escalation team: "
    f"{len(final_escalation_demo_cases):,} rows."
)